In [0]:
import pytest
from pyspark.sql import SparkSession
from pyspark.sql.functions import col


# =========================================================
# SPARK SESSION
# =========================================================
@pytest.fixture(scope="session")
def spark():
    return SparkSession.builder \
        .appName("DLT Testing") \
        .getOrCreate()


# =========================================================
# DIM TABLE TESTS
# =========================================================

def test_dim_applicant(spark):
    df = spark.table("LIVE.dim_applicant")

    assert df.count() > 0
    assert df.filter(col("applicant_id").isNull()).count() == 0


def test_dim_loan(spark):
    df = spark.table("LIVE.dim_loan")

    assert df.count() > 0
    assert df.filter(col("applicant_id").isNull()).count() == 0


def test_dim_credit(spark):
    df = spark.table("LIVE.dim_credit")

    assert df.count() > 0
    assert df.filter(col("credit_score").isNull()).count() == 0


def test_dim_economic(spark):
    df = spark.table("LIVE.dim_economic")

    assert df.count() > 0
    assert df.filter(col("region").isNull()).count() == 0


def test_dim_time(spark):
    df = spark.table("LIVE.dim_time")

    assert df.count() > 0
    assert df.filter(col("year").isNull()).count() == 0


# =========================================================
# FACT TABLE TESTS
# =========================================================

def test_fact_not_empty(spark):
    df = spark.table("LIVE.fact_loan_application")

    assert df.count() > 0, "❌ FACT TABLE EMPTY"


def test_fact_keys(spark):
    df = spark.table("LIVE.fact_loan_application")

    assert df.filter(
        col("applicant_key").isNull() |
        col("loan_key").isNull() |
        col("credit_key").isNull()
    ).count() == 0


def test_fact_measures(spark):
    df = spark.table("LIVE.fact_loan_application")

    assert df.filter(col("loan_amount").isNull()).count() == 0


def test_ltv_valid(spark):
    df = spark.table("LIVE.fact_loan_application")

    assert df.filter(col("LTV") < 0).count() == 0


def test_risk_category_valid(spark):
    df = spark.table("LIVE.fact_loan_application")

    valid = ["High", "Medium", "Low"]

    assert df.filter(~col("risk_category").isin(valid)).count() == 0


# =========================================================
# JOIN VALIDATION (IMPORTANT)
# =========================================================

def test_fact_join_integrity(spark):
    fact = spark.table("LIVE.fact_loan_application")
    dim = spark.table("LIVE.dim_applicant")

    missing = fact.join(
        dim,
        fact.applicant_key == dim.applicant_key,
        "left_anti"
    ).count()

    assert missing == 0, "❌ Join mismatch found"